# 📝 Генератор документации модели

Этот ноутбук создает полную документацию для построенной модели:

1. **README.md** - основная документация компании с описанием всех данных и конфигурации
2. **MODEL_SPECIFICATION.md** - детальная спецификация модели:
   - Параметры моделирования из истории
   - Расчетные параметры
   - Методы моделирования статей
   - Допущения моделирования
   - Трансформации данных
3. **YAML_PARAMETERS.md** - все параметры конфигурации в удобном виде

## 🚀 Быстрый старт

1. Выберите компанию в следующей ячейке
2. Запустите все ячейки последовательно
3. Документация будет создана в директории компании


## 📚 Документация о Standard vs Custom режимах

При генерации документации учитывайте информацию о режимах моделирования:

- **docs/MODEL_ENGINES_COMPARISON.md** - детальное сравнение Standard vs Custom
- **docs/ENGINE_CONFIGURATION_MATRIX_ANALYSIS.md** - соответствие Engine Configuration Matrix
- **docs/IMPLEMENTATION_PLAN_STANDARD_CUSTOM.md** - план реализации

### Ключевые концепции:
- Standard Mode: упрощенная методология с fallback методами
- Custom Mode: продвинутая методология с полными schedules
- Все параметры рассчитываются в движке из истории
- Макродвижок доступен обоим режимам


In [ ]:
# Импорты и настройка
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from datetime import datetime
import sys
import subprocess

# Настройка пути
ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT))

print(f"📁 ROOT: {ROOT}")

# Список компаний
COMPANIES = [d.name for d in (ROOT / 'companies').iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f"📊 Доступные компании: {COMPANIES}")


def load_company_csv(relative_path: str) -> pd.DataFrame:
    """Показывает предупреждение, если используется legacy CSV."""
    csv_path = ROOT / relative_path
    if csv_path.exists():
        print(f"⚠️ Используем legacy CSV: {csv_path.relative_to(ROOT)}")
        try:
            return pd.read_csv(csv_path)
        except Exception as exc:
            print(f"⚠️ Ошибка чтения {relative_path}: {exc}")
            return pd.DataFrame()
    print(f"ℹ️ Legacy CSV отсутствует: {relative_path}")
    return pd.DataFrame()


In [ ]:
# Выбор компании
COMPANY = "rusal"  # Измените на нужную компанию

croot = ROOT / "companies" / COMPANY
proj_yaml_path = croot / "configs" / "project.yaml"

print(f"🏢 Компания: {COMPANY.upper()}")
print(f"📁 Путь: {croot}")
print(f"📄 Config: {proj_yaml_path.exists()}")


## Генерация документации

Запускаем скрипт генерации документации


In [ ]:
# Запуск генератора документации
!cd {ROOT} && python3 tools/generate_model_documentation.py --company {COMPANY}


## Просмотр созданной документации


In [ ]:
# Показываем созданные файлы
from IPython.display import display, Markdown

readme_path = croot / "README.md"
spec_path = croot / "MODEL_SPECIFICATION.md"
yaml_params_path = croot / "YAML_PARAMETERS.md"

print("✅ Созданные файлы документации:")
print(f"   1. README.md ({readme_path.stat().st_size:,} bytes)")
print(f"   2. MODEL_SPECIFICATION.md ({spec_path.stat().st_size:,} bytes)")
print(f"   3. YAML_PARAMETERS.md ({yaml_params_path.stat().st_size:,} bytes)")

# Если нужны CSV отпечатки (legacy), выводим предупреждение
legacy_files = {
    f"companies/{COMPANY}/history/is_history_{COMPANY}.csv": "IS History",
    f"companies/{COMPANY}/history/bs_history_{COMPANY}.csv": "BS History",
    f"companies/{COMPANY}/history/cf_history_{COMPANY}.csv": "CF History",
}
for relative_path, title in legacy_files.items():
    df = load_company_csv(relative_path)
    if not df.empty:
        display(Markdown(f"### ♻️ {title} (legacy CSV предупреждение)"))

# Показываем первые строки README
if readme_path.exists():
    with open(readme_path, 'r', encoding='utf-8') as f:
        readme_preview = ''.join(f.readlines()[:50])
    display(Markdown("### Превью README.md:"))
    display(Markdown(readme_preview))
    display(Markdown("**...** (файл обрезан для превью, полный файл доступен по пути выше)"))
